# 1.3 Row Filtering and Limiting ? Oracle HR Practice

**Hands-on PySpark notebook | Oracle HR sample-schema domain**

All records are synthetic and follow Oracle HR table conventions. The reusable HR DataFrames
include valid keys, NULLs, a missing department assignment, and duplicate employee email data
for realistic workforce filtering and sampling exercises.

**Learning objectives**
- Filter HR records using business and data-quality predicates.
- Return deterministic top-N results by sorting before limiting or collecting rows.
- Distinguish DataFrame-returning limits from row-collection actions.
- Use reproducible random and stratified samples for HR audits and analysis.

## 1. PySpark environment setup

Target: PySpark 3.5+ with a compatible Java runtime. All questions use the Oracle HR sample
schema DataFrames; Pandas is not used.

## 2. Required imports

In [ ]:
from datetime import date
from decimal import Decimal
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DateType, DecimalType, IntegerType, StringType, StructField, StructType

## 3. SparkSession creation

In [ ]:
spark = (SparkSession.builder.appName("Oracle HR DataFrame Creation and Structure Practice")
         .master("local[2]").getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

## 4. Create reusable Oracle HR DataFrames

EMPLOYEES and DEPARTMENTS contain 20 and 18 records. Related JOBS, JOB_HISTORY, LOCATIONS,
COUNTRIES, and REGIONS records are included. Primary keys are unique and present foreign keys
point to corresponding records; optional NULL relationships and duplicate email data are intentional.

In [ ]:
region_schema = StructType([StructField("region_id", IntegerType(), False), StructField("region_name", StringType(), False)])
regions_df = spark.createDataFrame([
    (1, "Europe"), (2, "Americas"), (3, "Asia"), (4, "Middle East and Africa"), (5, "Oceania")
], region_schema)

country_schema = StructType([
    StructField("country_id", StringType(), False), StructField("country_name", StringType(), False),
    StructField("region_id", IntegerType(), False)
])
countries_df = spark.createDataFrame([
    ("US", "United States of America", 2), ("CA", "Canada", 2), ("BR", "Brazil", 2),
    ("GB", "United Kingdom", 1), ("DE", "Germany", 1), ("FR", "France", 1),
    ("IN", "India", 3), ("JP", "Japan", 3), ("AU", "Australia", 5), ("EG", "Egypt", 4)
], country_schema)

location_schema = StructType([
    StructField("location_id", IntegerType(), False), StructField("street_address", StringType(), True),
    StructField("postal_code", StringType(), True), StructField("city", StringType(), False),
    StructField("state_province", StringType(), True), StructField("country_id", StringType(), False)
])
locations_df = spark.createDataFrame([
    (1000, "200 Market Street", "94105", "San Francisco", "California", "US"),
    (1001, "500 Fifth Avenue", "10018", "New York", "New York", "US"),
    (1002, "77 Massachusetts Avenue", "02139", "Cambridge", "Massachusetts", "US"),
    (1003, "10 King Street", "M5H 1A1", "Toronto", "Ontario", "CA"),
    (1004, "120 Rua Augusta", "01305-000", "Sao Paulo", "Sao Paulo", "BR"),
    (1005, "25 Canada Square", "E14 5LQ", "London", None, "GB"),
    (1006, "Unter den Linden 5", "10117", "Berlin", None, "DE"),
    (1007, "8 Rue de Rivoli", "75001", "Paris", None, "FR"),
    (1008, "14 Residency Road", "560025", "Bengaluru", "Karnataka", "IN"),
    (1009, "6 Connaught Place", "110001", "New Delhi", "Delhi", "IN"),
    (1010, "33 Marine Drive", "400020", "Mumbai", "Maharashtra", "IN"),
    (1011, "1 Chome Marunouchi", "100-0005", "Tokyo", None, "JP"),
    (1012, "88 George Street", "2000", "Sydney", "New South Wales", "AU"),
    (1013, "22 Nile Corniche", None, "Cairo", None, "EG"),
    (1014, "400 Lake Shore Drive", "60611", "Chicago", "Illinois", "US"),
    (1015, "9 Market Street", "94105", "San Francisco", "California", "US"),
    (1016, "70 Fleet Street", "EC4Y 1EU", "London", None, "GB"),
    (1017, "15 Park Street", None, "Kolkata", "West Bengal", "IN")
], location_schema)

job_schema = StructType([
    StructField("job_id", StringType(), False), StructField("job_title", StringType(), False),
    StructField("min_salary", DecimalType(8, 2), False), StructField("max_salary", DecimalType(8, 2), False)
])
jobs_df = spark.createDataFrame([
    ("AD_PRES", "President", Decimal("20000.00"), Decimal("40000.00")),
    ("AD_VP", "Administration Vice President", Decimal("15000.00"), Decimal("30000.00")),
    ("IT_PROG", "Programmer", Decimal("4000.00"), Decimal("12000.00")),
    ("FI_MGR", "Finance Manager", Decimal("8000.00"), Decimal("16000.00")),
    ("FI_ACCOUNT", "Accountant", Decimal("4000.00"), Decimal("10000.00")),
    ("PU_MAN", "Purchasing Manager", Decimal("8000.00"), Decimal("15000.00")),
    ("PU_CLERK", "Purchasing Clerk", Decimal("2000.00"), Decimal("5000.00")),
    ("SA_MAN", "Sales Manager", Decimal("10000.00"), Decimal("20000.00")),
    ("SA_REP", "Sales Representative", Decimal("5000.00"), Decimal("12000.00")),
    ("HR_REP", "Human Resources Representative", Decimal("4000.00"), Decimal("9000.00")),
    ("HR_MGR", "Human Resources Manager", Decimal("8000.00"), Decimal("15000.00")),
    ("MK_MAN", "Marketing Manager", Decimal("9000.00"), Decimal("15000.00")),
    ("MK_REP", "Marketing Representative", Decimal("5000.00"), Decimal("11000.00")),
    ("ST_MAN", "Stock Manager", Decimal("5000.00"), Decimal("10000.00")),
    ("ST_CLERK", "Stock Clerk", Decimal("2000.00"), Decimal("5000.00")),
    ("AC_MGR", "Accounting Manager", Decimal("8000.00"), Decimal("16000.00")),
    ("AC_ACCOUNT", "Public Accountant", Decimal("4000.00"), Decimal("10000.00")),
    ("PR_REP", "Public Relations Representative", Decimal("5000.00"), Decimal("11000.00"))
], job_schema)

employee_schema = StructType([
    StructField("employee_id", IntegerType(), False), StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), False), StructField("email", StringType(), True),
    StructField("phone_number", StringType(), True), StructField("hire_date", DateType(), False),
    StructField("job_id", StringType(), False), StructField("salary", DecimalType(8, 2), False),
    StructField("commission_pct", DecimalType(4, 2), True), StructField("manager_id", IntegerType(), True),
    StructField("department_id", IntegerType(), True)
])
employees_df = spark.createDataFrame([
    (100, "Olivia", "Bennett", "OBENNETT", "515.123.1000", date(1987,6,17), "AD_PRES", Decimal("24000.00"), None, None, 90),
    (101, "Marcus", "Chen", "MCHEN", "515.123.1001", date(1989,9,21), "AD_VP", Decimal("17000.00"), None, 100, 90),
    (102, "Priya", "Nair", "PNAIR", "515.123.1002", date(1993,1,13), "AD_VP", Decimal("17000.00"), None, 100, 90),
    (103, "Daniel", "Brooks", "DBROOKS", "590.423.1003", date(1990,1,3), "IT_PROG", Decimal("9000.00"), None, 102, 60),
    (104, "Sofia", "Patel", "SPATEL", "590.423.1004", date(1991,5,21), "IT_PROG", Decimal("6000.00"), None, 103, 60),
    (105, "Ethan", "Morris", "EMORRIS", "590.423.1005", date(1997,6,25), "IT_PROG", Decimal("4800.00"), None, 103, 60),
    (106, "Aisha", "Khan", "AKHAN", "590.423.1006", date(1998,2,5), "IT_PROG", Decimal("4800.00"), None, 103, 60),
    (107, "Lucas", "Meyer", "LMEYER", "590.423.1007", date(1999,2,7), "IT_PROG", Decimal("4200.00"), None, 103, 60),
    (108, "Grace", "Kim", "GKIM", "515.124.1008", date(1994,8,17), "FI_MGR", Decimal("12000.00"), None, 101, 100),
    (109, "Noah", "Williams", "NWILLIAMS", "515.124.1009", date(1994,8,16), "FI_ACCOUNT", Decimal("9000.00"), None, 108, 100),
    (110, "Elena", "Garcia", "EGARCIA", "515.124.1010", date(1997,9,28), "FI_ACCOUNT", Decimal("8200.00"), None, 108, 100),
    (111, "Ibrahim", "Hassan", "IHASSAN", "515.124.1011", date(1997,9,30), "FI_ACCOUNT", Decimal("7700.00"), None, 108, 100),
    (112, "Chloe", "Dubois", "CDUBOIS", "515.124.1012", date(1998,3,7), "FI_ACCOUNT", Decimal("7800.00"), None, 108, 100),
    (113, "Mateo", "Silva", "MSILVA", "515.124.1013", date(1999,12,7), "FI_ACCOUNT", Decimal("6900.00"), None, 108, 100),
    (114, "Amara", "Okafor", "AOKAFOR", "515.127.1014", date(1994,12,7), "PU_MAN", Decimal("11000.00"), None, 100, 30),
    (115, "Theo", "Martin", "TMARTIN", "515.127.1015", date(1995,5,18), "PU_CLERK", Decimal("3100.00"), None, 114, 30),
    (116, "Mei", "Tanaka", "MTANAKA", "515.127.1016", date(1997,12,24), "SA_MAN", Decimal("11000.00"), Decimal("0.15"), 100, 80),
    (117, "Fatima", "Rahman", "FRAHMAN", "515.127.1017", date(1997,7,24), "SA_REP", Decimal("8000.00"), Decimal("0.10"), 116, 80),
    (118, "Fatima", "Rahman", "FRAHMAN", "515.127.1018", date(1997,7,24), "SA_REP", Decimal("8000.00"), Decimal("0.10"), 116, 80),
    (119, "Ben", "Carter", None, None, date(1999,8,10), "HR_REP", Decimal("2500.00"), None, 100, None)
], employee_schema)

department_schema = StructType([
    StructField("department_id", IntegerType(), False), StructField("department_name", StringType(), False),
    StructField("manager_id", IntegerType(), True), StructField("location_id", IntegerType(), True)
])
departments_df = spark.createDataFrame([
    (10,"Administration",101,1000), (20,"Marketing",103,1001), (30,"Purchasing",114,1003),
    (40,"Human Resources",119,1008), (50,"Shipping",114,1009), (60,"IT",103,1002),
    (70,"Public Relations",101,1005), (80,"Sales",116,1014), (90,"Executive",100,1000),
    (100,"Finance",108,1006), (110,"Accounting Operations",109,1007), (120,"Recruiting",119,1017),
    (130,"Compensation",101,1008), (140,"Business Intelligence",103,1015),
    (150,"Customer Support",114,1016), (160,"Risk Management",108,1001),
    (170,"Legal",101,1005), (180,"Facilities",None,None)
], department_schema)

job_history_schema = StructType([
    StructField("employee_id", IntegerType(), False), StructField("start_date", DateType(), False),
    StructField("end_date", DateType(), False), StructField("job_id", StringType(), False),
    StructField("department_id", IntegerType(), True)
])
job_history_df = spark.createDataFrame([
    (101,date(1987,9,21),date(1989,9,20),"AC_ACCOUNT",110),
    (102,date(1989,1,13),date(1992,12,31),"IT_PROG",60),
    (103,date(1987,1,3),date(1989,12,31),"SA_REP",80),
    (104,date(1989,5,21),date(1991,5,20),"IT_PROG",60),
    (105,date(1994,6,25),date(1997,6,24),"IT_PROG",60),
    (108,date(1990,8,17),date(1994,8,16),"FI_ACCOUNT",100),
    (109,date(1990,8,16),date(1994,8,15),"AC_ACCOUNT",110),
    (110,date(1994,9,28),date(1997,9,27),"AC_ACCOUNT",110),
    (114,date(1990,12,7),date(1994,12,6),"PU_CLERK",30),
    (115,date(1992,5,18),date(1995,5,17),"PU_CLERK",30),
    (116,date(1993,12,24),date(1997,12,23),"SA_REP",80),
    (117,date(1994,7,24),date(1997,7,23),"SA_REP",80),
    (118,date(1994,7,24),date(1997,7,23),"SA_REP",80),
    (119,date(1995,8,10),date(1999,8,9),"HR_REP",40),
    (101,date(1985,1,1),date(1987,9,20),"AC_ACCOUNT",110)
], job_history_schema)

hr_dataframes = {
    "employees_df": employees_df, "departments_df": departments_df,
    "jobs_df": jobs_df, "job_history_df": job_history_df,
    "locations_df": locations_df, "countries_df": countries_df, "regions_df": regions_df
}
print("Prepared reusable HR DataFrames:", ", ".join(hr_dataframes.keys()))


## 5. Preview sample records and schemas

Review sample records and declared schemas before attempting the exercises. These same seven HR
DataFrames are reused throughout the notebook.

In [ ]:
for dataframe_name, dataframe in hr_dataframes.items():
    print(f"\n--- {dataframe_name}: sample rows ---")
    dataframe.show(3, truncate=False)
    print(f"--- {dataframe_name}: schema ---")
    dataframe.printSchema()

## 6. Practical questions

Each question is one Markdown prompt followed by one completely empty code cell. There are three
practical tasks for each of the seven listed concepts.

## filter()

### Question 1 ? Easy

**Business scenario:** The IT workforce coordinator needs a current roster of employees assigned to the IT department.

**Problem statement:** Use `filter()` on `employees_df` to keep employees whose department ID is 60. Return employee ID, first name, last name, job ID, and department ID.

**Expected output:** Five rows are returned for employees 103?107 with the requested columns; no other departments appear.

### Question 2 ? Medium

**Business scenario:** Compensation operations needs to review well-paid employees who are not receiving a sales commission.

**Problem statement:** Use `filter()` to find employees with salary at least 9000 and a NULL commission percentage. Return employee ID, job ID, salary, and commission percentage.

**Expected output:** Seven employees match: 100, 101, 102, 103, 108, 109, and 114; all returned commission percentages are NULL.

### Question 3 ? Hard

**Business scenario:** HR master-data controls must identify employees with incomplete manager, department, or commission attributes without discarding source rows.

**Problem statement:** Use `filter()` with a compound condition to return employees missing at least one of manager ID, department ID, or commission percentage. Include employee ID and all three checked fields.

**Expected output:** Seventeen employees match; employees 116?118 are not included because all three checked fields are present. Employee 100 has no manager and employee 119 has no department.

## where()

### Question 4 ? Easy

**Business scenario:** The finance department needs an employee list for its monthly headcount reconciliation.

**Problem statement:** Use `where()` on `employees_df` to return employees assigned to department 100 with employee ID, name, job ID, and salary.

**Expected output:** Six rows are returned for employees 108?113, with the requested fields.

### Question 5 ? Medium

**Business scenario:** A workforce analyst needs employees in IT or Finance earning at least 8000 per month.

**Problem statement:** Use `where()` with combined predicates to filter department 60 or 100 and salary of at least 8000. Return employee ID, department ID, job ID, and salary.

**Expected output:** Five employees match: 103, 108, 109, 110, and 112; all satisfy the department and salary conditions.

### Question 6 ? Hard

**Business scenario:** A migration exception report must catch employees whose email or department assignment is missing.

**Problem statement:** Use `where()` with NULL-safe predicates to return employee ID, email, manager ID, and department ID when email is NULL or department ID is NULL.

**Expected output:** One record is returned: employee 119, whose email and department ID are NULL; no valid employee is lost due to NULL comparison behavior.

## limit()

### Question 7 ? Easy

**Business scenario:** An HR directory preview needs the three highest-paid employees, with ties handled consistently.

**Problem statement:** Order `employees_df` by salary descending and employee ID ascending, then use `limit()` to return employee ID, name, and salary for the first three.

**Expected output:** The ordered result contains employees 100, 101, and 102 with salaries 24000.00, 17000.00, and 17000.00.

### Question 8 ? Medium

**Business scenario:** The IT onboarding team needs the two earliest hires from the IT department.

**Problem statement:** Filter employees to department 60, order by hire date ascending and employee ID ascending, then use `limit()` to return the first two employee records.

**Expected output:** Two rows are returned: employee 103 hired 1990-01-03 and employee 104 hired 1991-05-21.

### Question 9 ? Hard

**Business scenario:** Sales compensation audit requires a deterministic top-two list among employees who receive commission.

**Problem statement:** Filter out NULL commission percentages, order by commission percentage descending, then salary descending and employee ID ascending, and apply `limit(2)`.

**Expected output:** The result contains employee 116 first (0.15 commission) and employee 117 second (0.10 commission, tie resolved by employee ID).

## head()

### Question 10 ? Easy

**Business scenario:** An HR data steward wants a deterministic three-record spot check from the employee master.

**Problem statement:** Order employees by employee ID and use `head()` to retrieve the first three records.

**Expected output:** A Python list of three Row objects is returned for employee IDs 100, 101, and 102.

### Question 11 ? Medium

**Business scenario:** The HR catalog owner needs a small, sorted preview of job definitions.

**Problem statement:** Order `jobs_df` by job ID and use `head(4)` to retrieve four job records.

**Expected output:** A list of four Row objects is returned with job IDs AC_ACCOUNT, AC_MGR, AD_PRES, and AD_VP in that order.

### Question 12 ? Hard

**Business scenario:** A manager needs the two highest-paid IT employees for an audit sample, ordered deterministically.

**Problem statement:** Filter employees to department 60, order by salary descending and employee ID ascending, and use `head(2)`.

**Expected output:** A list of two Row objects is returned: employee 103 at 9000.00 and employee 104 at 6000.00.

## take()

### Question 13 ? Easy

**Business scenario:** A migration engineer needs four employee records in key order for a connectivity check.

**Problem statement:** Order `employees_df` by employee ID and use `take(4)`.

**Expected output:** A Python list of four Row objects is returned for employee IDs 100, 101, 102, and 103.

### Question 14 ? Medium

**Business scenario:** Sales operations wants the two highest-commission employees, with a stable tie-breaker.

**Problem statement:** Filter employees to non-NULL commission percentage, order by commission descending and employee ID ascending, and use `take(2)`.

**Expected output:** Two Row objects are returned for employees 116 and 117, with commission percentages 0.15 and 0.10.

### Question 15 ? Hard

**Business scenario:** Job-history migration validation needs the three most recently ended department-80 assignments, including deterministic handling of same-day end dates.

**Problem statement:** Filter `job_history_df` to department 80, order by end date descending and employee ID ascending, and use `take(3)`.

**Expected output:** Three Row objects are returned for employees 116, 117, and 118, ordered by end dates 1997-12-23, 1997-07-23, and 1997-07-23.

## sample()

### Question 16 ? Easy

**Business scenario:** The HR analytics team needs a repeatable small employee sample for a pipeline smoke test.

**Problem statement:** Use `sample()` without replacement on `employees_df` with fraction 0.25 and a fixed seed. Inspect the returned employee IDs and row count.

**Expected output:** A subset of the employee master is returned with the same schema; repeating the call with the same seed and runtime returns the same sampled rows. The fraction is probabilistic, not an exact row-count guarantee.

### Question 17 ? Medium

**Business scenario:** A data-quality analyst wants to inspect a reproducible sample of employee records without duplicating employees.

**Problem statement:** Use `sample()` without replacement with fraction 0.5 and a fixed seed, then validate that sampled employee IDs are unique and the sample is smaller than the full 20-row source.

**Expected output:** A reproducible subset with the employee schema is returned; no employee ID is repeated and at most 20 rows are selected; the exact count is not guaranteed by the fraction.

### Question 18 ? Hard

**Business scenario:** An HR statistician is building a bootstrap-style resample where employee records may appear more than once.

**Problem statement:** Use `sample()` with replacement and a fixed seed to produce a resample from `employees_df`; report total sampled rows and employee IDs that occur more than once.

**Expected output:** A reproducible resampled DataFrame with the original schema is returned; total rows need not equal 20, and repeated employee IDs are allowed by the sampling design.

## sampleBy()

### Question 19 ? Easy

**Business scenario:** IT and Finance leaders need a reproducible stratified employee sample for a workforce review.

**Problem statement:** Use `sampleBy()` with department ID as the strata column, fraction 0.5 for departments 60 and 100, and a fixed seed. Inspect the resulting department IDs and counts.

**Expected output:** Only departments 60 and 100 appear; the sample is reproducible with the fixed seed, and each stratum count is an approximate sample rather than a guaranteed exact half.

### Question 20 ? Medium

**Business scenario:** The HR migration team needs a complete sample of IT and Sales employees plus a partial Finance sample.

**Problem statement:** Use `sampleBy()` on `employees_df` stratified by job ID with fractions 1.0 for IT_PROG and SA_REP, and 0.5 for FI_ACCOUNT. Use a fixed seed and validate counts per job.

**Expected output:** All five IT_PROG employees and both SA_REP employees are included; FI_ACCOUNT is sampled at approximately half; other job strata are absent.

### Question 21 ? Hard

**Business scenario:** A compensation audit requires deterministic stratified selection across job families while retaining all commission-bearing sales roles.

**Problem statement:** Use `sampleBy()` stratified by job ID with fraction 1.0 for SA_MAN and SA_REP, 0.4 for FI_ACCOUNT, and 0.2 for IT_PROG; set a fixed seed. Validate that all sales-role rows remain, other job strata are excluded, and repeated runs produce the same employee IDs.

**Expected output:** All three sales rows (employees 116?118) are included; only a reproducible subset of FI_ACCOUNT and IT_PROG is present; no other job families appear. Sampling counts for fractional strata are approximate.

## 7. Final practice completion checklist

- [ ] I completed all 21 question cells independently.
- [ ] I applied deterministic ordering before requesting top-N or collected rows.
- [ ] I accounted for NULL values when constructing filters.
- [ ] I used fixed seeds when comparing random or stratified samples.
- [ ] I preserved HR source records and key relationships during analysis.

After completing your work, stop Spark with `spark.stop()` in your own cell.